
---------------------------------------------------------------------------------------------------------------
**count(lit(1))** counts every row because the literal **1** is never NULL.
**count(col("acctId"))** counts only rows where acctId is **not NULL.**

custId	count(lit(1))	count(acctId)
1001	2	            2
1002	1	            1
1004	1	            0

----------------------------------------------------------------------------------------------------------------

### **What is Medallion Architecture?**
- Medallion Architecture is a layered data design pattern used in Databricks, where data moves through three layers: **BRONZE--> SILVER--> GOLD**. 
- Bronze stores raw data, Silver focuses on cleaning and validating the data, and Gold provides business-ready datasets for reporting and analytics. 
- The main idea is to progressively **improve data quality and usability as data moves** through these layers. 

### **What belongs in Bronze?**
Bronze is the **raw ingestion layer**. 
We bring data from different source systems and store it as **close to its original format as possible.** 
We may also capture **metadata** like ingestion timestamps, source file names, and batch IDs for auditing and troubleshooting

### **What belongs in Silver?**
- Silver is where we **clean and validate the raw data**. 
- We handle duplicates, null values, data type conversions, and invalid records. 
- We can also **standardize fields and join related datasets.** 
- The goal is to create clean, reliable data that downstream processes can use.

Silver is the cleaned and validated layer in Medallion Architecture. This is where we handle duplicates, null values, and data quality checks. For example, in a customer data pipeline, we can remove duplicate customer records and separate orphan accounts before making the data available for downstream processing.

### **What belongs in Gold?**
- Gold is the** business-ready layer.** 
- This is where we **_apply business logic, create aggregations, and build datasets for specific reporting or analytical requirements. _**
- For example, we might create customer-level summaries, revenue metrics, or account balances that Power BI dashboards can consume directly.

### **Why shouldn't we perform heavy cleaning in Bronze?**
We generally avoid heavy cleaning in Bronze because we want to preserve the original source data. 
If there's an issue with our transformation logic or business rules change, we can **reprocess the raw data and rebuild** the Silver and Gold layers without having to extract everything again from the source systems.

### **Why retain Bronze after Silver has been created?**
We retain Bronze **for auditing, troubleshooting, and reprocessing. **
For example, if we discover that our Silver transformation incorrectly filtered some records, we can go back to Bronze, correct the logic, and rebuild the affected datasets. 
It also helps us trace data back to its original source.

One useful distinction: Bronze retention is not necessarily indefinite. Retention policies depend on storage costs, compliance requirements, and the ability to replay source data.

### **Where should deduplication happen?**
- Deduplication typically **happens in Silver.** 
- We preserve the raw records in Bronze and apply deduplication rules in Silver based on business keys or timestamps. For example, if we receive multiple records for the same customer, we can use a window function to retain the latest valid record.

### **Where should business aggregations usually happen?**
- **Business aggregations usually happen in Gold** because that's the layer designed for reporting and analytics. 
- Silver focuses on producing clean, validated data, while Gold applies business-specific calculations like total revenue, customer account counts, or monthly sales summaries.

**_Business-facing metrics generally happens in GOLD_**

### **What happens if Gold needs to be rebuilt?**
- If Gold needs to be rebuilt, we can **reprocess the data from Silver **because Silver already contains clean and validated datasets. 
- For example, if a business calculation changes, we can update the Gold transformation and regenerate the Gold tables without reprocessing Bronze, as long as the Silver data has everything we need.

### **How is Medallion Architecture different from a traditional staging → EDW → data mart architecture?**
Both architectures follow a similar concept of moving data through different stages. 
In a traditional data warehouse, 
    **staging** --> holds incoming data,  
    **EDW** --> integrates and organizes enterprise data, 
    **data marts **--> serve specific business areas.

Medallion Architecture follows a similar layered approach, but it's commonly implemented in a lakehouse environment like Databricks. 
**Bronze** --> preserves raw data, 
**Silver** --> provides cleaned and integrated data, and 
**Gold** --> provides business-ready datasets. 
The main difference is that Medallion Architecture works with lakehouse technologies that **support both structured and semi-structured data**, along with scalable processing and transactional capabilities through Delta Lake.


### **Senior scenario**
You receive customer data from CRM, account data from a billing platform, and usage data from Kafka. Design a Bronze/Silver/Gold architecture that supports auditability, data-quality checks and BI reporting.


----------------------------------------------------------------------------------------------------------------------------------------------------
Suppose customer 1001 has two accounts:

**Accounts table**
````
custId	acctId	balance
1001	5001	100
1001	5002	50
````

**Payments table**
````
paymentId	acctId	paymentAmount
P1	5001	10
P2	5001	20
P3	5001	15
P4	5001	25
P5	5001	30
P6	5002	5
P7	5002	10
P8	5002	15
P9	5002	10
P10	5002	10
````

## **2. What happens after joining?**
````
JOINED RESULT — 10 ROWS
custId	acctId	balance	paymentId
1001	5001	100	P1
1001	5001	100	P2
1001	5001	100	P3
1001	5001	100	P4
1001	5001	100	P5
1001	5002	50	P6
1001	5002	50	P7
1001	5002	50	P8
1001	5002	50	P9
1001	5002	50	P10
````

Notice that each account balance is repeated five times because each account has five matching payments.

Now your aggregation:
````
groupBy("custId").agg(
    count("acctId").alias("accountCount"),
    sum("balance").alias("totalBalance")
)
````

Produces:
````
**custId	accountCount	totalBalance**
1001	10	750
````

But the correct business result is:
````
**custId	accountCount	totalBalance**
1001	2	150
````

The balance is inflated because (100 × 5) + (50 × 5) = 750.

**3. Would adding acctId to groupBy fix it?**
You suggested grouping by both custId and acctId.

That would produce two rows, as you expected, but the balances would still be inflated.
````
**custId	acctId	count(acctId)	sum(balance)**
1001	5001	5	500
1001	5002	5	250
````

So grouping by _acctId_ changes the level of aggregation but doesn't eliminate duplicate account balances introduced by the join.

**4. How should a senior data engineer fix this?**
The safest approach is to **_aggregate each dataset at its appropriate grain before joining._**

For example, calculate customer account metrics directly from the accounts table:
````
dfAccountSummary = (
    dfAccounts
    .groupBy("custId")
    .agg(
        **countDistinct**("acctId").alias("accountCount"),
        sum("balance").alias("totalBalance")
    )
)
````

Separately aggregate payments if the business needs payment metrics:
````
dfPaymentSummary = (
    dfPayments
    .join(
        dfAccounts.select("acctId", "custId"),
        "acctId",
        "inner"
    )
    .groupBy("custId")
    .agg(
        sum("paymentAmount").alias("totalPayments")
    )
)
````

Then**_ join the two customer-level summaries using custId._**

This assumes the accounts table has one row per account. If it contains duplicate account records, validate or deduplicate them first.

**### Certification takeaway: **
**countDistinct(acctId)** can correct an inflated account count, but it does not correct an inflated balance sum. 
And **sumDistinct(balance)** is not a safe fix because different accounts can legitimately have identical balances.


### **5. One more interview question**
You have an Accounts table:
````
acctId	custId	balance
5001	1001	100
5002	1001	100
````

After joining with Payments, each account appears three times.

_Would this fix the total balance?_ ----> **IT WILL RETURN 100**
````
groupBy("custId").agg(
    sumDistinct("balance")
)
````
Because it considers **distinct balance values**, not distinct account IDs.
Both accounts have a balance of 100, so Spark sees only one distinct value to sum

### **CORRECT APPROACH:**
Aggregate the accounts before joining with Payments:
````
dfAccountSummary = (
    dfAccounts
    .groupBy("custId")
    .agg(
        count("acctId").alias("accountCount"),
        sum("balance").alias("totalBalance")
    )
)
````

**This produces the correct total of 200, assuming one record per account.**

**Remember this interview rule:**
- **sum() **adds every non-null value, including duplicates.
- **sumDistinct()** adds each unique non-null value only once.
- Neither function automatically understands your business key (acctId).
The proper solution is to control the grain of your data before joining and aggregating.

After joining with Payments, each account appears three times, producing six rows.
- sum(balance) = 600 (incorrect, inflated by the join)
- sumDistinct(balance) = 100 (incorrect, because both accounts have the same balance)
- Actual total balance = 200
sumDistinct() eliminates duplicate values, not duplicate records based on acctId.

----------------------------------------------------------------------------------------------------------------------------------------------------

### **Suppose you have:**
- 100 customers
- 250 accounts
- 1,000 payments
You perform an INNER JOIN across all three tables.

**Can you determine the exact number of output rows from these counts alone? If not, what additional information would you need?**

Assume the relationship are:
````
Customers (custId)
      │
      │ 1-to-many
      ▼
Accounts (acctId, custId)
      │
      │ 1-to-many
      ▼
Payments (paymentId, acctId)
````

We would needs details on - 
1. **Orphan accounts:** Accounts whose custId doesn't exist in Customers.
2. **Orphan payments:** Payments whose acctId doesn't exist in Accounts.
3. **Accounts without payments: **These accounts disappear from the final INNER JOIN.
4. **Join cardinality:** Whether customer IDs and account IDs are unique in their respective master tables. Duplicate keys can multiply the resulting rows.

Example
Assuming unique customer IDs, unique account IDs, and each payment belongs to at most one account:
````
**Scenario	                                                                                      Final rows**
All 1,000 payments have valid accounts and customers	                                            1,000
100 payments have no matching account	                                                            900
100 payments belong to accounts with no matching customer	                                        900
100 payments have no matching account, and another 50 belong to accounts with no matching customer	850
````

The exact number **depends on how many payments successfully match through both relationships.**

### **Quick follow-up question
### Suppose:**
- 100 customers
- 250 accounts
- 1,000 payments
- Every account belongs to a valid customer.
- Every payment belongs to a valid account.
- Customer IDs and account IDs are unique in their master tables.
How many rows will the final INNER JOIN produce?

A. 100
B. 250
**C. 1,000**
D. 1,350

**Why 1,000 rows?**

Given our assumptions:
- 100 customers, each with a unique custId.
- 250 accounts, each with a unique acctId and a valid customer.
- 1,000 payments, each referencing a valid account.
The joins behave as follows:
````
**Operation	                    Output rows**
Customers INNER JOIN Accounts	250
Result INNER JOIN Payments	    1,000
````

Why? Each payment matches exactly one account, and each account matches exactly one customer. Therefore, every payment produces exactly one output row.

The final result **contains 1,000 rows, not necessarily 1,000 unique accounts or customers.**

This is because the final dataset is at the** payment grain: one row per payment.**

